In [1]:
"""
================================================================================
 PROYEK AKHIR - PEMBELAJARAN MESIN DAN KECERDASAN BUATAN
================================================================================
 Tema   : Transformasi Digital Industri melalui Implementasi AI dan ML
 Topik  : Predictive Maintenance – Prediksi Kegagalan Mesin Industri
 Dataset: AI4I 2020 Predictive Maintenance Dataset (10.000 data poin)
 Sumber : UCI Machine Learning Repository
================================================================================
 Deskripsi:
   Proyek ini membangun sistem prediksi kegagalan mesin (Machine Failure)
   menggunakan dua pendekatan algoritma berbeda:
     1. Random Forest Classifier   (Scikit-Learn – Ensemble Method)
     2. MLP Neural Network         (Scikit-Learn – Deep Learning Approach)
   Perbandingan performa antar model dilakukan menggunakan F1-Score,
   Confusion Matrix, ROC-AUC, dan Stratified Cross-Validation.
================================================================================
"""

# ── Library Imports ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                   # non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, learning_curve)
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.ensemble        import RandomForestClassifier
from sklearn.neural_network  import MLPClassifier
from sklearn.metrics         import (classification_report, confusion_matrix,
                                     roc_auc_score, roc_curve, f1_score,
                                     ConfusionMatrixDisplay, precision_recall_curve,
                                     average_precision_score)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.inspection      import permutation_importance

import os, sys, time


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("stephanmatzka/predictive-maintenance-dataset-ai4i-2020")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'predictive-maintenance-dataset-ai4i-2020' dataset.
Path to dataset files: /kaggle/input/predictive-maintenance-dataset-ai4i-2020


In [5]:
# ── Path Setup ─────────────────────────────────────────────────────────────────
# Menggunakan path dari kagglehub yang sudah didownload sebelumnya
import os
import glob

# Mencari file csv secara otomatis di dalam folder hasil download kagglehub
csv_files = glob.glob(os.path.join(path, "*.csv"))
if csv_files:
    DATA_PATH = csv_files[0]
else:
    DATA_PATH = os.path.join(path, "ai4i2020.csv")

OUTPUT_DIR = "/content/outputs" # Diubah ke direktori lokal Colab
os.makedirs(OUTPUT_DIR, exist_ok=True)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"Dataset path yang digunakan: {DATA_PATH}")

# ── Colour Palette ─────────────────────────────────────────────────────────────
PALETTE = {
    "primary": "#2563EB",
    "secondary": "#DC2626",
    "accent": "#16A34A",
    "neutral": "#6B7280",
    "bg": "#F8FAFC",
    "warn": "#D97706"
}
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.facecolor": PALETTE["bg"],
    "axes.facecolor": PALETTE["bg"],
    "font.family": "DejaVu Sans"
})

Dataset path yang digunakan: /kaggle/input/predictive-maintenance-dataset-ai4i-2020/ai4i2020.csv


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. DATA LOADING & INITIAL INSPECTION
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print(" STEP 1 – DATA LOADING & INITIAL INSPECTION")
print("=" * 70)

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

print(f"\n▸ Shape           : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"▸ Missing values  : {df.isnull().sum().sum()}")
print(f"▸ Duplicate rows  : {df.duplicated().sum()}")
print("\n── Column dtypes ──────────────────────────────────────")
print(df.dtypes.to_string())
print("\n── Descriptive Statistics ─────────────────────────────")
print(df.describe().round(3).to_string())

# ── Class Distribution ─────────────────────────────────────────────────────────
fail_counts = df["Machine failure"].value_counts()
fail_pct    = df["Machine failure"].value_counts(normalize=True) * 100
print(f"\n── Target (Machine Failure) Distribution ──────────────")
print(f"  No Failure  (0): {fail_counts[0]:,}  ({fail_pct[0]:.2f}%)")
print(f"  Failure     (1): {fail_counts[1]:,}  ({fail_pct[1]:.2f}%)")
print(f"\n  → Imbalance ratio  ≈  {fail_counts[0]/fail_counts[1]:.1f}:1")
print(f"  → Class imbalance ditangani dengan class_weight='balanced'\n")

# ── Failure Type Breakdown ─────────────────────────────────────────────────────
failure_types = ["TWF", "HDF", "PWF", "OSF", "RNF"]
failure_labels = {
    "TWF": "Tool Wear Failure",
    "HDF": "Heat Dissipation Failure",
    "PWF": "Power Failure",
    "OSF": "Overstrain Failure",
    "RNF": "Random Failure"
}
print("── Failure Sub-Type Counts ────────────────────────────")
for ft in failure_types:
    cnt = df[ft].sum()
    pct = cnt / len(df) * 100
    print(f"  {failure_labels[ft]:<30}: {cnt:>3} ({pct:.2f}%)")

 STEP 1 – DATA LOADING & INITIAL INSPECTION

▸ Shape           : 10,000 rows × 14 columns
▸ Missing values  : 0
▸ Duplicate rows  : 0

── Column dtypes ──────────────────────────────────────
UDI                          int64
Product ID                  object
Type                        object
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64

── Descriptive Statistics ─────────────────────────────
             UDI  Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure        TWF        HDF        PWF        OSF        RNF
count  10000.000            10000.000                10000.000           

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. EXPLORATORY DATA ANALYSIS (EDA)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(" STEP 2 – EXPLORATORY DATA ANALYSIS (EDA)")
print("=" * 70)

numerical_features = ["Air temperature [K]", "Process temperature [K]",
                       "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]

# ── Figure 1 : Overview Dashboard ─────────────────────────────────────────────
fig = plt.figure(figsize=(20, 22), facecolor=PALETTE["bg"])
fig.suptitle("EDA Dashboard – AI4I 2020 Predictive Maintenance",
             fontsize=18, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.5, wspace=0.4)

# 1a. Target distribution (pie + bar)
ax1 = fig.add_subplot(gs[0, 0])
colors_pie = [PALETTE["accent"], PALETTE["secondary"]]
wedges, texts, autotexts = ax1.pie(
    fail_counts.values,
    labels=["No Failure\n(96.6%)", "Failure\n(3.4%)"],
    autopct="%1.1f%%",
    colors=colors_pie, startangle=90,
    textprops={"fontsize": 10}, pctdistance=0.75)
ax1.set_title("Target Class Distribution", fontweight="bold")

# 1b. Failure sub-types bar chart
ax2 = fig.add_subplot(gs[0, 1])
ft_counts = [df[ft].sum() for ft in failure_types]
bar_colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["warn"],
              PALETTE["accent"], PALETTE["neutral"]]
bars = ax2.barh([failure_labels[ft] for ft in failure_types], ft_counts,
                color=bar_colors, edgecolor="white", height=0.6)
for bar, cnt in zip(bars, ft_counts):
    ax2.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
             str(cnt), va="center", fontsize=9, fontweight="bold")
ax2.set_xlabel("Count")
ax2.set_title("Failure Sub-Type Distribution", fontweight="bold")

# 1c. Product type distribution
ax3 = fig.add_subplot(gs[0, 2])
type_counts  = df["Type"].value_counts()
type_colors  = [PALETTE["primary"], PALETTE["accent"], PALETTE["warn"]]
ax3.bar(type_counts.index, type_counts.values,
        color=type_colors, edgecolor="white", width=0.6)
for i, (t, c) in enumerate(zip(type_counts.index, type_counts.values)):
    ax3.text(i, c + 50, str(c), ha="center", fontweight="bold")
ax3.set_title("Product Type Distribution", fontweight="bold")
ax3.set_xlabel("Type (L=Low, M=Medium, H=High)")
ax3.set_ylabel("Count")

# 1d–1h. Histograms per numerical feature
for idx, feat in enumerate(numerical_features):
    ax = fig.add_subplot(gs[1, idx % 3] if idx < 3 else gs[2, idx - 3])
    df[df["Machine failure"] == 0][feat].hist(
        ax=ax, bins=40, color=PALETTE["accent"], alpha=0.65, label="No Failure", density=True)
    df[df["Machine failure"] == 1][feat].hist(
        ax=ax, bins=40, color=PALETTE["secondary"], alpha=0.65, label="Failure", density=True)
    ax.set_title(f"{feat}", fontweight="bold", fontsize=9)
    ax.set_xlabel(feat, fontsize=8)
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)

# 1i. Correlation Heatmap
ax_corr = fig.add_subplot(gs[2, 2])
corr_cols = numerical_features + ["Machine failure"]
corr_matrix = df[corr_cols].corr()
sns.heatmap(corr_matrix, ax=ax_corr, annot=True, fmt=".2f",
            cmap="RdYlGn", linewidths=0.5, annot_kws={"size": 7},
            vmin=-1, vmax=1, square=True)
ax_corr.set_title("Feature Correlation Heatmap", fontweight="bold", fontsize=9)
ax_corr.tick_params(axis="x", rotation=45, labelsize=7)
ax_corr.tick_params(axis="y", rotation=0, labelsize=7)

# 1j. Box plots – Torque vs Failure
ax_box1 = fig.add_subplot(gs[3, 0])
df.boxplot(column="Torque [Nm]", by="Machine failure", ax=ax_box1,
           patch_artist=True,
           boxprops=dict(facecolor=PALETTE["primary"], alpha=0.7),
           medianprops=dict(color="white", linewidth=2))
ax_box1.set_title("Torque vs Failure", fontweight="bold")
ax_box1.set_xlabel("Machine Failure (0=No, 1=Yes)")
plt.sca(ax_box1); plt.title("Torque [Nm] vs Machine Failure")

# 1k. Box plots – Tool wear vs Failure
ax_box2 = fig.add_subplot(gs[3, 1])
df.boxplot(column="Tool wear [min]", by="Machine failure", ax=ax_box2,
           patch_artist=True,
           boxprops=dict(facecolor=PALETTE["warn"], alpha=0.7),
           medianprops=dict(color="white", linewidth=2))
ax_box2.set_title("Tool Wear vs Failure", fontweight="bold")
plt.sca(ax_box2); plt.title("Tool Wear [min] vs Machine Failure")

# 1l. Scatter – Rotational Speed vs Torque
ax_sc = fig.add_subplot(gs[3, 2])
fail_mask = df["Machine failure"] == 1
ax_sc.scatter(df.loc[~fail_mask, "Rotational speed [rpm]"],
              df.loc[~fail_mask, "Torque [Nm]"],
              c=PALETTE["accent"], alpha=0.3, s=5, label="No Failure")
ax_sc.scatter(df.loc[fail_mask, "Rotational speed [rpm]"],
              df.loc[fail_mask, "Torque [Nm]"],
              c=PALETTE["secondary"], alpha=0.7, s=10, label="Failure")
ax_sc.set_xlabel("Rotational Speed [rpm]")
ax_sc.set_ylabel("Torque [Nm]")
ax_sc.set_title("Speed vs Torque (by Failure)", fontweight="bold")
ax_sc.legend(fontsize=8)

eda_path = os.path.join(OUTPUT_DIR, "eda_dashboard.png")
plt.savefig(eda_path, dpi=150, bbox_inches="tight", facecolor=PALETTE["bg"])
plt.close()
print(f"\n✔ EDA dashboard saved → {eda_path}")


 STEP 2 – EXPLORATORY DATA ANALYSIS (EDA)

✔ EDA dashboard saved → /content/outputs/eda_dashboard.png


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. FEATURE ENGINEERING & PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(" STEP 3 – FEATURE ENGINEERING & PREPROCESSING")
print("=" * 70)

df_ml = df.copy()

# 3a. Encode 'Type' (L=0, M=1, H=2)
le = LabelEncoder()
df_ml["Type_enc"] = le.fit_transform(df_ml["Type"])
print(f"\n▸ LabelEncoder mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# 3b. Derived features (domain-relevant)
df_ml["Temp_diff"]        = df_ml["Process temperature [K]"] - df_ml["Air temperature [K]"]
df_ml["Power"]            = (df_ml["Torque [Nm]"] *
                              df_ml["Rotational speed [rpm]"] * 2 * np.pi / 60)  # Watts
df_ml["Wear_Torque"]      = df_ml["Tool wear [min]"] * df_ml["Torque [Nm]"]
df_ml["Speed_Torque"]     = df_ml["Rotational speed [rpm]"] * df_ml["Torque [Nm]"]

print("▸ Derived features created:")
print("   • Temp_diff     = Process_temp − Air_temp  (thermal gradient)")
print("   • Power         = Torque × ω               (mechanical power in Watts)")
print("   • Wear_Torque   = Tool_wear × Torque        (wear-load interaction)")
print("   • Speed_Torque  = Speed × Torque            (power proxy)")

# 3c. Feature selection
FEATURE_COLS = [
    "Type_enc",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Temp_diff",
    "Power",
    "Wear_Torque",
    "Speed_Torque"
]
TARGET_COL = "Machine failure"

X = df_ml[FEATURE_COLS].values
y = df_ml[TARGET_COL].values

print(f"\n▸ Feature matrix X  : {X.shape}")
print(f"▸ Target vector  y  : {y.shape}")

# 3d. Train / Test split (stratified, 80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

# 3e. Standard Scaling (applied only to numerical features)
scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"\n▸ Train set : {X_train.shape[0]:,} samples")
print(f"▸ Test set  : {X_test.shape[0]:,} samples")
print(f"▸ Train failure rate: {y_train.mean()*100:.2f}%")
print(f"▸ Test  failure rate: {y_test.mean()*100:.2f}%")

# ── Class weights (to handle imbalance) ───────────────────────────────────────
cw_vals = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: cw_vals[0], 1: cw_vals[1]}
print(f"\n▸ Class weights (balanced): {class_weight_dict}")


 STEP 3 – FEATURE ENGINEERING & PREPROCESSING

▸ LabelEncoder mapping: {'H': np.int64(0), 'L': np.int64(1), 'M': np.int64(2)}
▸ Derived features created:
   • Temp_diff     = Process_temp − Air_temp  (thermal gradient)
   • Power         = Torque × ω               (mechanical power in Watts)
   • Wear_Torque   = Tool_wear × Torque        (wear-load interaction)
   • Speed_Torque  = Speed × Torque            (power proxy)

▸ Feature matrix X  : (10000, 10)
▸ Target vector  y  : (10000,)

▸ Train set : 8,000 samples
▸ Test set  : 2,000 samples
▸ Train failure rate: 3.39%
▸ Test  failure rate: 3.40%

▸ Class weights (balanced): {0: np.float64(0.51753137533963), 1: np.float64(14.760147601476016)}


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. MODEL A – RANDOM FOREST CLASSIFIER
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(" STEP 4A – MODEL 1: RANDOM FOREST CLASSIFIER")
print("=" * 70)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

t0 = time.time()
rf_model.fit(X_train, y_train)
rf_train_time = time.time() - t0

y_pred_rf       = rf_model.predict(X_test)
y_prob_rf       = rf_model.predict_proba(X_test)[:, 1]

rf_f1   = f1_score(y_test, y_pred_rf, average="binary")
rf_auc  = roc_auc_score(y_test, y_prob_rf)

print(f"\n▸ Training time    : {rf_train_time:.2f}s")
print(f"▸ F1-Score (test)  : {rf_f1:.4f}")
print(f"▸ ROC-AUC (test)   : {rf_auc:.4f}")
print("\n── Classification Report (Random Forest) ──────────────")
print(classification_report(y_test, y_pred_rf,
                             target_names=["No Failure", "Failure"]))

# Stratified Cross-Validation (RF)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_f1_rf = cross_val_score(rf_model, X_train, y_train,
                             cv=skf, scoring="f1", n_jobs=-1)
print(f"▸ 5-Fold CV F1  : {cv_f1_rf.mean():.4f} ± {cv_f1_rf.std():.4f}")



 STEP 4A – MODEL 1: RANDOM FOREST CLASSIFIER

▸ Training time    : 3.39s
▸ F1-Score (test)  : 0.8750
▸ ROC-AUC (test)   : 0.9704

── Classification Report (Random Forest) ──────────────
              precision    recall  f1-score   support

  No Failure       0.99      1.00      1.00      1932
     Failure       0.93      0.82      0.88        68

    accuracy                           0.99      2000
   macro avg       0.96      0.91      0.94      2000
weighted avg       0.99      0.99      0.99      2000

▸ 5-Fold CV F1  : 0.8618 ± 0.0450


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. MODEL B – MLP NEURAL NETWORK (Scikit-Learn)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(" STEP 4B – MODEL 2: MLP NEURAL NETWORK (Deep Learning Approach)")
print("=" * 70)

mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),   # 3-layer deep network
    activation="relu",
    solver="adam",
    alpha=1e-3,                         # L2 regularization
    batch_size=64,
    learning_rate="adaptive",
    learning_rate_init=1e-3,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=RANDOM_STATE
)

t0 = time.time()
mlp_model.fit(X_train_sc, y_train)
mlp_train_time = time.time() - t0

y_pred_mlp      = mlp_model.predict(X_test_sc)
y_prob_mlp      = mlp_model.predict_proba(X_test_sc)[:, 1]

mlp_f1  = f1_score(y_test, y_pred_mlp, average="binary")
mlp_auc = roc_auc_score(y_test, y_prob_mlp)

print(f"\n▸ Architecture         : Input(10) → Dense(128) → Dense(64) → Dense(32) → Output(1)")
print(f"▸ Activation           : ReLU (hidden layers)")
print(f"▸ Optimizer            : Adam  |  LR = 0.001 (adaptive)")
print(f"▸ Regularization       : L2 (alpha=0.001)  +  Early Stopping")
print(f"▸ Iterations (actual)  : {mlp_model.n_iter_}")
print(f"▸ Training time        : {mlp_train_time:.2f}s")
print(f"▸ F1-Score (test)      : {mlp_f1:.4f}")
print(f"▸ ROC-AUC (test)       : {mlp_auc:.4f}")
print("\n── Classification Report (MLP Neural Network) ─────────")
print(classification_report(y_test, y_pred_mlp,
                             target_names=["No Failure", "Failure"]))

cv_f1_mlp = cross_val_score(mlp_model, X_train_sc, y_train,
                              cv=skf, scoring="f1", n_jobs=-1)
print(f"▸ 5-Fold CV F1  : {cv_f1_mlp.mean():.4f} ± {cv_f1_mlp.std():.4f}")



 STEP 4B – MODEL 2: MLP NEURAL NETWORK (Deep Learning Approach)

▸ Architecture         : Input(10) → Dense(128) → Dense(64) → Dense(32) → Output(1)
▸ Activation           : ReLU (hidden layers)
▸ Optimizer            : Adam  |  LR = 0.001 (adaptive)
▸ Regularization       : L2 (alpha=0.001)  +  Early Stopping
▸ Iterations (actual)  : 88
▸ Training time        : 16.17s
▸ F1-Score (test)      : 0.7402
▸ ROC-AUC (test)       : 0.9723

── Classification Report (MLP Neural Network) ─────────
              precision    recall  f1-score   support

  No Failure       0.99      0.99      0.99      1932
     Failure       0.80      0.69      0.74        68

    accuracy                           0.98      2000
   macro avg       0.89      0.84      0.87      2000
weighted avg       0.98      0.98      0.98      2000

▸ 5-Fold CV F1  : 0.6899 ± 0.0536


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# 6. EVALUATION & COMPARISON PLOTS
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(" STEP 5 – EVALUATION, VALIDATION & COMPARISON")
print("=" * 70)

fig2, axes = plt.subplots(3, 3, figsize=(20, 18), facecolor=PALETTE["bg"])
fig2.suptitle("Model Evaluation & Comparison – Predictive Maintenance",
              fontsize=16, fontweight="bold", y=0.98)

# ── 6a. Confusion Matrix – RF ──────────────────────────────────────────────────
ax = axes[0, 0]
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(cm_rf, display_labels=["No Failure", "Failure"])
disp_rf.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix\nRandom Forest", fontweight="bold")

# ── 6b. Confusion Matrix – MLP ─────────────────────────────────────────────────
ax = axes[0, 1]
cm_mlp = confusion_matrix(y_test, y_pred_mlp)
disp_mlp = ConfusionMatrixDisplay(cm_mlp, display_labels=["No Failure", "Failure"])
disp_mlp.plot(ax=ax, colorbar=False, cmap="Oranges")
ax.set_title("Confusion Matrix\nMLP Neural Network", fontweight="bold")

# ── 6c. ROC Curves – both models ───────────────────────────────────────────────
ax = axes[0, 2]
fpr_rf,  tpr_rf,  _ = roc_curve(y_test, y_prob_rf)
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, y_prob_mlp)
ax.plot(fpr_rf,  tpr_rf,  color=PALETTE["primary"],
        lw=2, label=f"Random Forest (AUC={rf_auc:.3f})")
ax.plot(fpr_mlp, tpr_mlp, color=PALETTE["secondary"],
        lw=2, label=f"MLP Network  (AUC={mlp_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random Baseline")
ax.fill_between(fpr_rf,  tpr_rf,  alpha=0.08, color=PALETTE["primary"])
ax.fill_between(fpr_mlp, tpr_mlp, alpha=0.08, color=PALETTE["secondary"])
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison", fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ── 6d. Precision–Recall Curves ────────────────────────────────────────────────
ax = axes[1, 0]
prec_rf,  rec_rf,  _ = precision_recall_curve(y_test, y_prob_rf)
prec_mlp, rec_mlp, _ = precision_recall_curve(y_test, y_prob_mlp)
ap_rf  = average_precision_score(y_test, y_prob_rf)
ap_mlp = average_precision_score(y_test, y_prob_mlp)
ax.plot(rec_rf,  prec_rf,  color=PALETTE["primary"],
        lw=2, label=f"RF  (AP={ap_rf:.3f})")
ax.plot(rec_mlp, prec_mlp, color=PALETTE["secondary"],
        lw=2, label=f"MLP (AP={ap_mlp:.3f})")
baseline = y_test.sum() / len(y_test)
ax.axhline(y=baseline, color="gray", linestyle="--", lw=1, label=f"Baseline ({baseline:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision–Recall Curve", fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ── 6e. Feature Importances (RF) ───────────────────────────────────────────────
ax = axes[1, 1]
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=True)
colors_fi   = [PALETTE["secondary"] if imp > importances.median()
               else PALETTE["primary"] for imp in importances.values]
importances.plot(kind="barh", ax=ax, color=colors_fi, edgecolor="white")
ax.set_title("Feature Importance\n(Random Forest – MDI)", fontweight="bold")
ax.set_xlabel("Mean Decrease in Impurity")

# ── 6f. Cross-Validation F1 Comparison ─────────────────────────────────────────
ax = axes[1, 2]
cv_data = pd.DataFrame({
    "Fold":         [f"Fold {i+1}" for i in range(5)] * 2,
    "F1-Score":     list(cv_f1_rf) + list(cv_f1_mlp),
    "Model":        ["Random Forest"] * 5 + ["MLP Network"] * 5
})
colors_cv = [PALETTE["primary"], PALETTE["secondary"]]
x_pos = np.arange(5)
w = 0.35
ax.bar(x_pos - w/2, cv_f1_rf,  w, label="Random Forest", color=PALETTE["primary"],
       alpha=0.8, edgecolor="white")
ax.bar(x_pos + w/2, cv_f1_mlp, w, label="MLP Network",   color=PALETTE["secondary"],
       alpha=0.8, edgecolor="white")
ax.set_xticks(x_pos)
ax.set_xticklabels([f"Fold {i+1}" for i in range(5)])
ax.set_ylabel("F1-Score")
ax.set_title("5-Fold Cross-Validation F1", fontweight="bold")
ax.legend(fontsize=9)
ax.axhline(y=cv_f1_rf.mean(),  linestyle="--", color=PALETTE["primary"],   alpha=0.6, lw=1)
ax.axhline(y=cv_f1_mlp.mean(), linestyle="--", color=PALETTE["secondary"], alpha=0.6, lw=1)

# ── 6g. Learning Curve – RF ────────────────────────────────────────────────────
ax = axes[2, 0]
train_sizes, train_scores, val_scores = learning_curve(
    rf_model, X_train, y_train,
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    scoring="f1", n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 8))
ts_mean = train_scores.mean(axis=1); ts_std = train_scores.std(axis=1)
vs_mean = val_scores.mean(axis=1);   vs_std = val_scores.std(axis=1)
ax.plot(train_sizes, ts_mean, "o-", color=PALETTE["primary"],  label="Train F1")
ax.plot(train_sizes, vs_mean, "s-", color=PALETTE["secondary"], label="Val F1")
ax.fill_between(train_sizes, ts_mean-ts_std, ts_mean+ts_std, alpha=0.1, color=PALETTE["primary"])
ax.fill_between(train_sizes, vs_mean-vs_std, vs_mean+vs_std, alpha=0.1, color=PALETTE["secondary"])
ax.set_xlabel("Training Set Size")
ax.set_ylabel("F1-Score")
ax.set_title("Learning Curve – Random Forest", fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ── 6h. MLP Loss Curve ────────────────────────────────────────────────────────
ax = axes[2, 1]
ax.plot(mlp_model.loss_curve_, color=PALETTE["primary"],   lw=2, label="Train Loss")
if hasattr(mlp_model, "validation_scores_") and mlp_model.validation_scores_ is not None:
    # Convert validation_score (accuracy) proxy
    val_loss_proxy = [1 - s for s in mlp_model.validation_scores_]
    ax.plot(val_loss_proxy, color=PALETTE["secondary"], lw=2,
            linestyle="--", label="Val Loss (proxy)")
ax.set_xlabel("Iteration (Epoch)"); ax.set_ylabel("Loss")
ax.set_title("MLP Training Loss Curve", fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ── 6i. Model Comparison Bar Chart ────────────────────────────────────────────
ax = axes[2, 2]
metrics_names = ["F1-Score", "ROC-AUC", "CV-F1 Mean"]
rf_scores  = [rf_f1,  rf_auc,  cv_f1_rf.mean()]
mlp_scores = [mlp_f1, mlp_auc, cv_f1_mlp.mean()]

x_idx = np.arange(len(metrics_names))
w = 0.35
bars_rf  = ax.bar(x_idx - w/2, rf_scores,  w, label="Random Forest",
                  color=PALETTE["primary"], alpha=0.85, edgecolor="white")
bars_mlp = ax.bar(x_idx + w/2, mlp_scores, w, label="MLP Network",
                  color=PALETTE["secondary"], alpha=0.85, edgecolor="white")
for bar in list(bars_rf) + list(bars_mlp):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.set_xticks(x_idx); ax.set_xticklabels(metrics_names)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Model Performance Comparison", fontweight="bold")
ax.legend(fontsize=9)

plt.tight_layout()
eval_path = os.path.join(OUTPUT_DIR, "model_evaluation.png")
plt.savefig(eval_path, dpi=150, bbox_inches="tight", facecolor=PALETTE["bg"])
plt.close()
print(f"✔ Evaluation plots saved → {eval_path}")


 STEP 5 – EVALUATION, VALIDATION & COMPARISON
✔ Evaluation plots saved → /content/outputs/model_evaluation.png


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# 7. FINAL RESULTS SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(" STEP 6 – FINAL RESULTS SUMMARY")
print("=" * 70)

summary_data = {
    "Metric":           ["F1-Score", "ROC-AUC", "AP-Score",
                          "CV-F1 (5-fold)", "Train Time (s)"],
    "Random Forest":    [f"{rf_f1:.4f}", f"{rf_auc:.4f}", f"{ap_rf:.4f}",
                          f"{cv_f1_rf.mean():.4f} ± {cv_f1_rf.std():.4f}",
                          f"{rf_train_time:.2f}"],
    "MLP Network":      [f"{mlp_f1:.4f}", f"{mlp_auc:.4f}", f"{ap_mlp:.4f}",
                          f"{cv_f1_mlp.mean():.4f} ± {cv_f1_mlp.std():.4f}",
                          f"{mlp_train_time:.2f}"],
}
summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

winner = "Random Forest" if rf_f1 >= mlp_f1 else "MLP Neural Network"
best_f1 = max(rf_f1, mlp_f1)
print(f"\n🏆 Best Model: {winner}  (F1 = {best_f1:.4f})")


 STEP 6 – FINAL RESULTS SUMMARY

        Metric   Random Forest     MLP Network
      F1-Score          0.8750          0.7402
       ROC-AUC          0.9704          0.9723
      AP-Score          0.8740          0.8023
CV-F1 (5-fold) 0.8618 ± 0.0450 0.6899 ± 0.0536
Train Time (s)            3.39           16.17

🏆 Best Model: Random Forest  (F1 = 0.8750)


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# 8. INDUSTRI INTERPRETATION (CPL 5)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(" STEP 7 – INTERPRETASI INDUSTRI & DAMPAK BISNIS")
print("=" * 70)

TP_rf = cm_rf[1, 1]; FN_rf = cm_rf[1, 0]; FP_rf = cm_rf[0, 1]
TP_mlp = cm_mlp[1, 1]; FN_mlp = cm_mlp[1, 0]; FP_mlp = cm_mlp[0, 1]
COST_UNPLANNED = 50_000_000   # IDR per kejadian (downtime tidak terencana)
COST_INSPECTION =  500_000   # IDR per false alarm

print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║                  ANALISIS DAMPAK BISNIS                              ║
╠══════════════════════════════════════════════════════════════════════╣
║  Asumsi Biaya:                                                       ║
║  • Biaya downtime tidak terencana (FN): Rp {COST_UNPLANNED:,}/kejadian  ║
║  • Biaya inspeksi false alarm     (FP): Rp {COST_INSPECTION:,}/kejadian     ║
╠══════════════════════════════════════════════════════════════════════╣
║  RANDOM FOREST:                                                      ║
║  • Missed failures (FN): {FN_rf:>3}  → Rp {FN_rf * COST_UNPLANNED:>15,} potensi kerugian  ║
║  • False alarms    (FP): {FP_rf:>3}  → Rp {FP_rf * COST_INSPECTION:>15,} biaya inspeksi   ║
║  • Total cost proxy     : Rp {FN_rf*COST_UNPLANNED + FP_rf*COST_INSPECTION:>15,}                   ║
╠══════════════════════════════════════════════════════════════════════╣
║  MLP NEURAL NETWORK:                                                 ║
║  • Missed failures (FN): {FN_mlp:>3}  → Rp {FN_mlp * COST_UNPLANNED:>15,} potensi kerugian  ║
║  • False alarms    (FP): {FP_mlp:>3}  → Rp {FP_mlp * COST_INSPECTION:>15,} biaya inspeksi   ║
║  • Total cost proxy     : Rp {FN_mlp*COST_UNPLANNED + FP_mlp*COST_INSPECTION:>15,}                   ║
╚══════════════════════════════════════════════════════════════════════╝
""")

print("""
TEMUAN KUNCI DARI EDA & PEMODELAN:
───────────────────────────────────────────────────────────────────────
1. FITUR PALING BERPENGARUH (dari Feature Importance RF):
   • Torque [Nm]         – torsi tinggi meningkatkan risiko kerusakan
   • Tool wear [min]     – akumulasi keausan pahat yang kritis
   • Power (derived)     – daya mekanik sebagai indikator beban lebih
   • Rotational speed    – kecepatan di luar rentang optimal = bahaya

2. POLA KEGAGALAN:
   • Heat Dissipation Failure (HDF) adalah tipe terbanyak
   • Kegagalan cenderung terjadi saat Torque > 60 Nm ATAU Wear > 200 min
   • Perbedaan suhu proses–udara yang kecil berkorelasi dengan HDF

3. REKOMENDASI SISTEM PREVENTIVE MAINTENANCE:
   a) Pasang threshold alert: Tool wear > 200 min → jadwalkan penggantian
   b) Monitor torsi secara real-time; alert jika > 65 Nm berkelanjutan
   c) Implementasikan sistem pendingin tambahan jika Temp_diff < 8.6 K
   d) Integrasikan model Random Forest ke SCADA/PLC untuk prediksi online

4. ESTIMASI ROI (berdasarkan test set):"""
f"""   • Model dapat mendeteksi ~{TP_rf} dari {TP_rf+FN_rf} kejadian gagal pada test set
   • Recall failure: {TP_rf/(TP_rf+FN_rf)*100:.1f}%  → mencegah kerugian downtime signifikan
   • Dengan 10.000 unit/tahun, potensi penghematan bisa mencapai
     ratusan juta rupiah per tahun dibanding corrective maintenance

KETERBATASAN ETIS & TEKNIS:
───────────────────────────────────────────────────────────────────────
• Dataset sintetis (AI4I 2020) – validasi dengan data produksi nyata wajib
• Model bias terhadap kondisi mesin tipe L (60% data)
• Threshold klasifikasi default (0.5) dapat disesuaikan sesuai risk appetite
• Perlu monitoring drift model setiap 3–6 bulan untuk menjaga akurasi
""")

print("=" * 70)
print(f"✔ SELESAI – Semua output tersimpan di: {OUTPUT_DIR}")
print("=" * 70)



 STEP 7 – INTERPRETASI INDUSTRI & DAMPAK BISNIS

╔══════════════════════════════════════════════════════════════════════╗
║                  ANALISIS DAMPAK BISNIS                              ║
╠══════════════════════════════════════════════════════════════════════╣
║  Asumsi Biaya:                                                       ║
║  • Biaya downtime tidak terencana (FN): Rp 50,000,000/kejadian  ║
║  • Biaya inspeksi false alarm     (FP): Rp 500,000/kejadian     ║
╠══════════════════════════════════════════════════════════════════════╣
║  RANDOM FOREST:                                                      ║
║  • Missed failures (FN):  12  → Rp     600,000,000 potensi kerugian  ║
║  • False alarms    (FP):   4  → Rp       2,000,000 biaya inspeksi   ║
║  • Total cost proxy     : Rp     602,000,000                   ║
╠══════════════════════════════════════════════════════════════════════╣
║  MLP NEURAL NETWORK:                                                 ║
║  • Missed failur